# PyTorch Learning Notebook

A first pass at PyTorch fundamentals: tensors, autograd, and a basic training loop.
The end-to-end example uses a toy regression problem (predicting a return-like target from a few features) since that maps onto the kind of data you already work with.

## 0. Check the install

In [ ]:
import torch
import torch.nn as nn

print("torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())  # False is expected on cpuonly builds

## 1. Tensors

Tensors are PyTorch's array type — like numpy arrays, but they can track gradients and run on GPU. Think of them as the pandas DataFrame/numpy array equivalent for this ecosystem.

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])
y = torch.ones(3)
print(x + y)
print(x * y)
print(x.shape, x.dtype)

In [ ]:
import numpy as np

# numpy <-> tensor conversion, common when mixing with pandas/numpy pipelines
arr = np.random.randn(4, 3)
t = torch.from_numpy(arr).float()
print(t)
print(t.numpy())

## 2. Autograd

Set `requires_grad=True` and PyTorch records the operations on a tensor so it can compute gradients automatically via `.backward()`. This is what makes gradient descent on a custom model possible without deriving the gradient by hand.

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

x = torch.tensor(3.0)
y = w * x + b          # y = 2*3 + 1 = 7
loss = (y - 10) ** 2    # squared error against a target of 10

loss.backward()
print("loss:", loss.item())
print("dloss/dw:", w.grad.item())
print("dloss/db:", b.grad.item())

## 3. A toy dataset

Synthetic data: target is a noisy linear combination of a few features, similar in spirit to a factor-model setup. The point here is the PyTorch mechanics, not the modeling realism.

In [ ]:
torch.manual_seed(0)

n_samples, n_features = 500, 4
X = torch.randn(n_samples, n_features)

true_weights = torch.tensor([0.5, -0.3, 0.8, 0.1])
true_bias = 0.05
noise = 0.1 * torch.randn(n_samples)

y = X @ true_weights + true_bias + noise
y = y.unsqueeze(1)  # shape (n_samples, 1) to match model output

print(X.shape, y.shape)

## 4. Define a model

`nn.Module` is the base class for any model. Here a single linear layer — equivalent to OLS, just fit by gradient descent instead of a closed-form solution.

In [ ]:
class LinearRegressor(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.linear = nn.Linear(n_features, 1)

    def forward(self, x):
        return self.linear(x)

model = LinearRegressor(n_features)
print(model)
print("initial weights:", model.linear.weight.data)

## 5. Training loop

The standard pattern: forward pass → compute loss → zero old gradients → backward pass → optimizer step. Every PyTorch training loop, no matter how complex the model, is a variation of this.

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

n_epochs = 200
losses = []

for epoch in range(n_epochs):
    optimizer.zero_grad()
    preds = model(X)
    loss = criterion(preds, y)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

    if epoch % 20 == 0:
        print(f"epoch {epoch:3d}  loss {loss.item():.4f}")

print("\nlearned weights:", model.linear.weight.data)
print("true weights:    ", true_weights)
print("learned bias:", model.linear.bias.item(), " true bias:", true_bias)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(losses)
plt.xlabel("epoch")
plt.ylabel("MSE loss")
plt.title("Training loss")
plt.show()

## 6. Next steps

- Swap `LinearRegressor` for a small MLP (`nn.Sequential` with `nn.ReLU` between linear layers) and see how it does on nonlinear targets.
- Use `torch.utils.data.DataLoader` for mini-batches instead of feeding the full dataset each step.
- Try `Adam` instead of `SGD` as the optimizer.
- Once comfortable, point the same pipeline at real features (e.g. lagged returns, spreads, ratings) instead of synthetic data.